In [1]:
from google.colab import files
uploaded = files.upload()

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
import pandas as pd
import re

contacts = pd.read_csv('tij_LH10.dat', sep=' ', header=None, names=['timestamp', 'person_A', 'person_B'])

# Adjust the name if your metadata file is called something else (e.g., metadata_Thiers13.dat)
meta = pd.read_csv('metadata_LH10.dat', header=None, names=['raw'])

# Extract person IDs and group labels
meta['person_id'] = meta['raw'].apply(lambda x: int(re.findall(r'\d+', str(x))[0]))  # Extract number
meta['group'] = meta['raw'].apply(lambda x: re.findall(r'[A-Za-z]+', str(x))[0] if re.findall(r'[A-Za-z]+', str(x)) else None)

# Merge group labels for both persons
contacts = contacts.merge(meta[['person_id', 'group']], left_on='person_A', right_on='person_id', how='left') \
                   .rename(columns={'group': 'group_A'}).drop(columns=['person_id'])

contacts = contacts.merge(meta[['person_id', 'group']], left_on='person_B', right_on='person_id', how='left') \
                   .rename(columns={'group': 'group_B'}).drop(columns=['person_id'])

# Standardize timestamps (SocioPatterns use 20s intervals)
contacts['time_seconds'] = contacts['timestamp'] * 20
contacts['time_minutes'] = contacts['time_seconds'] / 60
contacts['time_hours'] = contacts['time_seconds'] / 3600

# Save final cleaned dataset
output_csv = "tij__merged.csv"
contacts.to_csv(output_csv, index=False)
print(f"Cleaned and merged data saved as: {output_csv}")

# Download the result
files.download(output_csv)

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
from google.colab import files
import pandas as pd
import os

all_data = []

for filename in uploaded.keys():
    if filename.endswith(".csv"):
        print(f" Adding {filename}...")
        df = pd.read_csv(filename)
        df['source_file'] = os.path.splitext(filename)[0]
        all_data.append(df)

# STEP 3: Merge all datasets
merged_df = pd.concat(all_data, ignore_index=True)

In [ ]:
output_filename = "all_compiled_contacts.csv"
merged_df.to_csv(output_filename, index=False)

print(f"\n Successfully compiled {len(all_data)} datasets into '{output_filename}'")
files.download(output_filename)

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# === STEP 1: Load your compiled dataset ===
df = pd.read_csv("/content/all_compiled_contacts.csv")

In [ ]:
# === STEP 2: Drop duplicates and nulls ===
df = df.drop_duplicates()
df = df.dropna(subset=["timestamp", "person_A", "person_B"])

In [ ]:
# === STEP 3: Remove self-contacts ===
df = df[df["person_A"] != df["person_B"]]

In [ ]:
# === STEP 4: Convert datatypes ===
df["person_A"] = df["person_A"].astype(int)
df["person_B"] = df["person_B"].astype(int)
df["timestamp"] = df["timestamp"].astype(float)

In [ ]:
# === STEP 5: Convert timestamp to real datetime ===
# timestamps are in seconds from April 17, 2009
start_date = pd.to_datetime("2009-04-17 00:00:00")
df["datetime"] = start_date + pd.to_timedelta(df["timestamp"], unit="s")

# Separate into date and hour columns
df["date"] = df["datetime"].dt.date
df["hour"] = df["datetime"].dt.time

In [ ]:
# === STEP 6: Sort chronologically ===
df = df.sort_values(by="datetime").reset_index(drop=True)

In [ ]:
# === STEP 7: Normalize timestamp (optional, for modeling) ===
scaler = MinMaxScaler()
df["timestamp_norm"] = scaler.fit_transform(df[["timestamp"]])

In [ ]:
# === STEP 8: Save cleaned version ===
df.to_csv("final_contact_tracing_data.csv", index=False)
print("✅ Preprocessed data saved as 'preprocessed_contacts.csv'")

df.head()